In [20]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

Crear directorio `csv_cleaned`

In [21]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [22]:
Path(root / "csv_cleaned").mkdir(exist_ok=True)

Cargar archivos csv

In [23]:
df = pd.read_csv(root / "csv_enunciados/ejercicios_enunciados.csv", encoding="utf-8")

In [24]:
def clean_dataframe(df):
    df_sin_metadatos = df[df.aviso == "sin metadatos"]
    df = df[df.aviso.isna()]

    # modify ejercicio column
    df.ejercicio = df.ejercicio.apply(lambda x : x.replace("Opción", "opción")
                                      .replace(".", ",") if pd.notna(x) else x)

    # remove gravity from physics statements
    patron = r"g\s*=\s*9[,.]8\s*m\s*[·.]?\s*s\s*-\s*2\s*"
    mask = df.asignatura == "Física"
    df.loc[mask, "enunciado"] = df.loc[mask, "enunciado"].str.replace(patron, "", regex=True)
    
    # normalize text
    df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto)
                                      if pd.notna(texto) else texto)

    # trim multiple spaces
    df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto)
                                      .strip() if pd.notna(texto) else texto)

    # eliminar , al final del ejercicio
    df.ejercicio = df.ejercicio.str.strip(",")

    # create folder for clean csv
    output_dir = Path(root / "csv_cleaned")
    output_dir.mkdir(exist_ok=True)

    return df

Arreglar ejercicios de Ponencia

In [25]:
def fix_ponencia(row):
    if "PONENCIA" in row["info_ejercicio"]:
        row["convocatoria"] = "Ponencia"
        pos_ejercicio = row["info_ejercicio"].find("EJERCICIO")
        row["ejercicio"] = row["info_ejercicio"][pos_ejercicio:].title()
    return row

In [26]:
df = df.apply(fix_ponencia, axis = 1)

In [27]:
df[df.ejercicio.isna()]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,aviso
2556,Mates II,Mates II,2017,NaN,NaN,Matrices Y Determinantes,MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE...,Sea A una matriz 3x3 tal que det(2A)8.\na) ¿C...,2017 - Matrices Y Determinantes.pdf,NaN


In [28]:
df.loc[2556, "info_ejercicio"]

'MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE RUC CIC II O Ó 3 .N O PCIÓN B.'

In [29]:
df.loc[2556, ["convocatoria", "ejercicio"]] = ["Reserva 3", "Ejercicio 3, opción B"]

In [30]:
df.sample()

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,aviso
3680,Química,Química,2009,Septiembre,Ejercicio 6. Opción A,Ácido Base,QUÍMICA. 2009. SEPTIEMBRE. EJERCICIO 6. OPCIÓN A,En el laboratorio se tienen dos recipientes: u...,2009 - Ácido Base.pdf,NaN


In [31]:
df = clean_dataframe(df)

Tras la limpieza completa, creo el ID

In [32]:
def create_id(row):
    partes = [row["asignatura"], row["tema"], row["año"], row["convocatoria"]]
    if pd.notna(row["ejercicio"]):
        partes.append(row["ejercicio"])
    id_string = "_".join(str(p) for p in partes).lower()

    # quitar acentos: descomponer y eliminar los caracteres combinantes
    id_string = unicodedata.normalize("NFKD", id_string)
    id_string = "".join(c for c in id_string if not unicodedata.combining(c))

    # dejar solo letras, números y _ (quita comas, puntos, espacios...)
    id_string = re.sub(r"[^a-z0-9]+", "_", id_string).strip("_")
    return id_string

In [33]:
# create id
df["id"] = df.apply(lambda row: create_id(row), axis = 1)
    
# rearrange columns
df = df[["id"] + list(df.columns[:-2])]

Exportar el dataframe limpio

In [34]:
df.sample(5)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo
3177,quimica_conf_electronica_2002_septiembre_ejerc...,Química,Química,2002,Septiembre,"Ejercicio 2, opción B",Conf. Electrónica,QUÍMICA. 2002. SEPTIEMBRE. EJERCICIO 2. OPCIÓN B,Razone si las siguientes configuraciones elect...,2002 - Conf. Electrónica.pdf
623,mates_ccss_inferencia_estadistica_2004_reserva...,Mates CCSS,Mates CCSS,2004,Reserva 1,Ejercicio 3 Parte Ii opción B,Inferencia Estadística,SOCIALES II. 2004 RESERVA 1. EJERCICIO 3 PARTE...,Se sabe que la velocidad de los coches que cir...,2004 - Inferencia Estadística.pdf
2633,mates_ii_integrales_2019_reserva_3_ejercicio_2...,Mates II,Mates II,2019,Reserva 3,"Ejercicio 2, opción B",Integrales,MATEMÁTICAS II. 2019. RESERVA 3. EJERCICIO 2. ...,"  \nSea f : 0,  una función continua y se...",2019 - Integrales.pdf
3402,quimica_enlace_quimico_2006_junio_ejercicio_2_...,Química,Química,2006,Junio,"Ejercicio 2, opción A",Enlace Químico,QUÍMICA. 2006. JUNIO. EJERCICIO 2. OPCIÓN A,Dadas las moléculas de BCl y H O\n3 2\na) Dedu...,2006 - Enlace Químico.pdf
3063,quimica_formulacion_2001_reserva_2_ejercicio_1...,Química,Química,2001,Reserva 2,"Ejercicio 1, opción B",Formulación,QUÍMICA. 2001. RESERVA 2. EJERCICIO 1. OPCIÓN B,Formule o nombre los siguientes compuestos: a)...,2001 - Formulación.pdf


In [35]:
df.to_csv(root/"csv_cleaned/enunciados_ejercicios_clean.csv", index = False, encoding="utf-8")

In [36]:
df = pd.read_csv(root / "csv_cleaned/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [37]:
df.sample(3)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo
3265,quimica_termoquimica_2004_junio_ejercicio_6_op...,Química,Química,2004,Junio,"Ejercicio 6, opción A",Termoquímica,QUÍMICA. 2004. JUNIO. EJERCICIO 6. OPCIÓN A,"La nitroglicerina, C H (NO ) , se descompone s...",2004 - Termoquímica.pdf
561,mates_ccss_probabilidad_2002_reserva_4_ejercic...,Mates CCSS,Mates CCSS,2002,Reserva 4,"Ejercicio 3, Parte I, opción A",Probabilidad,S OCIALES II. 2002. RESERVA 4. EJERCICIO 3. PA...,S e dispone de una baraja española de 40 carta...,2002 - Probabilidad.pdf
762,mates_ccss_funciones_ccss_2007_reserva_2_ejerc...,Mates CCSS,Mates CCSS,2007,Reserva 2,"Ejercicio 2, opción B",Funciones CCSS,SOCIALES II. 2007. RESERVA 2. EJERCICIO 2. OPC...,Se considera la función f(x) x3 9x2 24x\na)...,2007 - Funciones CCSS.pdf
